<a href="https://colab.research.google.com/github/Anastasia-Marshakova/---/blob/main/1_mlops_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import os
from google.colab import userdata

for name in (
    "GITHUB_USER",
    "GITHUB_TOKEN",
    "DAGSHUB_USER",
    "DAGSHUB_TOKEN",
):
    os.environ[name] = userdata.get(name)

print("Все четыре секрета успешно прочитаны")

Все четыре секрета успешно прочитаны


In [26]:
REPO = "mlops-labs"

In [33]:
import os
import subprocess
from pathlib import Path

repo_path = Path("/content/mlops-labs")

if not repo_path.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Anastasia-Marshakova/mlops-labs.git",
            str(repo_path),
        ],
        check=True,
    )

if not (repo_path / ".git").is_dir():
    raise RuntimeError(
        "Папка mlops-labs существует, но не является Git-репозиторием."
    )

os.chdir(repo_path)

subprocess.run(
    ["git", "config", "user.name", "Anastasia-Marshakova"],
    check=True,
)
subprocess.run(
    [
        "git", "config", "user.email",
        "Anastasia-Marshakova@users.noreply.github.com",
    ],
    check=True,
)

print("Текущая папка:", Path.cwd())
subprocess.run(["git", "status"], check=True)

Текущая папка: /content/mlops-labs


CompletedProcess(args=['git', 'status'], returncode=0)

In [35]:
from pathlib import Path

folders = [
    "src",
    "notebooks",
    "tests",
    "data/external",
    "data/raw",
    "data/processed",
    "data/features",
    "models",
    "reports/plots",
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

Path("src/__init__.py").touch()

for folder in ("notebooks", "tests", "models", "reports"):
    Path(folder, ".gitkeep").touch()

print("Структура проекта создана")

Структура проекта создана


In [36]:
from pathlib import Path

path = Path(".gitignore")
existing = path.read_text(encoding="utf-8") if path.exists() else ""

rules = [
    "__pycache__/",
    "*.py[cod]",
    ".venv/",
    ".pytest_cache/",
    ".ruff_cache/",
    ".ipynb_checkpoints/",
    ".env",
    "data/external/",
]

lines = existing.splitlines()

for rule in rules:
    if rule not in lines:
        lines.append(rule)

path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(".gitignore настроен")

.gitignore настроен


In [37]:
!pwd
!git status --short

/content/mlops-labs
?? .gitignore
?? models/
?? notebooks/
?? reports/
?? src/
?? tests/


/content/название-репозитория

In [39]:
%%writefile pyproject.toml
[project]
name = "mlops-labs"
version = "0.1.0"
description = "MLOps labs: variant 10, Covertype classification"
requires-python = ">=3.12"
dependencies = [
    "numpy>=1.26",
    "pandas>=2.2",
    "scikit-learn>=1.5",
    "pyyaml>=6.0",
    "joblib>=1.4",
    "dvc[s3]>=3.55",
]

[dependency-groups]
dev = [
    "ruff>=0.7",
    "pre-commit>=4.0",
    "pytest>=8.3",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src"]

[tool.ruff]
line-length = 88
target-version = "py312"
extend-exclude = ["notebooks"]

[tool.ruff.lint]
select = ["E", "W", "F", "I", "B", "UP", "NPY", "PD"]

[tool.pytest.ini_options]
testpaths = ["tests"]

Writing pyproject.toml


In [45]:
import subprocess

subprocess.run(["git", "add", "-A"], check=True)

result = subprocess.run(
    ["pre-commit", "run", "--all-files"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)

print(result.stdout)

if result.returncode == 0:
    print("Все проверки пройдены.")
else:
    print(
        "Проверки требуют внимания. "
        "Если хуки исправили файлы — запусти эту ячейку ещё раз. "
        "Если ошибка повторяется — пришли вывод выше."
    )

check for added large files..............................................Passed
check yaml...............................................................Passed
detect private key.......................................................Passed
fix end of files.........................................................Passed
trim trailing whitespace.................................................Passed
ruff check...............................................................Passed
ruff format..............................................................Passed

Все проверки пройдены.


In [46]:
%cd /content/mlops-labs

/content


In [47]:
%%writefile .pre-commit-config.yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v6.0.0
    hooks:
      - id: check-added-large-files
        args: ["--maxkb=1024"]
      - id: check-yaml
      - id: detect-private-key
      - id: end-of-file-fixer
      - id: trailing-whitespace

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.17.3
    hooks:
      - id: ruff-check
        args: [--fix]
      - id: ruff-format

Overwriting .pre-commit-config.yaml


In [48]:
import subprocess

subprocess.run(["pre-commit", "install"], check=True)
subprocess.run(["git", "add", "-A"], check=True)
subprocess.run(["pre-commit", "run", "--all-files"], check=True)

CompletedProcess(args=['pre-commit', 'run', '--all-files'], returncode=0)

In [49]:
%cd /content/mlops-labs

import os
from google.colab import userdata

for name in ("DAGSHUB_USER", "DAGSHUB_TOKEN"):
    os.environ[name] = userdata.get(name)

print("Секреты DagsHub прочитаны")

/content
Секреты DagsHub прочитаны


In [50]:
import subprocess
from pathlib import Path

REPO = "mlops-labs"

def run_command(args):
    result = subprocess.run(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        message = result.stdout.replace(
            os.environ["DAGSHUB_TOKEN"], "[TOKEN HIDDEN]"
        )
        raise RuntimeError(message)

if not Path(".dvc").exists():
    run_command(["dvc", "init"])

run_command([
    "dvc", "remote", "add", "-d", "-f",
    "origin", "s3://dvc",
])

run_command([
    "dvc", "remote", "modify", "origin",
    "endpointurl",
    f"https://dagshub.com/{os.environ['DAGSHUB_USER']}/{REPO}.s3",
])

run_command([
    "dvc", "remote", "modify", "--local",
    "origin", "access_key_id",
    os.environ["DAGSHUB_TOKEN"],
])

run_command([
    "dvc", "remote", "modify", "--local",
    "origin", "secret_access_key",
    os.environ["DAGSHUB_TOKEN"],
])

print("DVC и хранилище настроены")

DVC и хранилище настроены


In [51]:
run_command(["git", "check-ignore", ".dvc/config.local"])

result = subprocess.run(
    ["dvc", "remote", "list"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)
print("Файл с ключами исключён из Git")

origin  s3://dvc        (default)

Файл с ключами исключён из Git


In [52]:
%cd /content/mlops-labs

/content


In [53]:
%%writefile params.yaml
ingest:
  cache_dir: data/external
  raw: data/raw/covertype.csv
  wilderness_column: Wilderness_Area_0
  n_rows: 10000
  seed: 42

data:
  target: Cover_Type
  test_size: 0.2
  split_seed: 42

Writing params.yaml


In [54]:
%%writefile src/ingest.py
"""Загрузка Covertype и создание воспроизводимого среза одной зоны."""

from pathlib import Path

import yaml
from sklearn.datasets import fetch_covtype


def main():
    params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))
    config = params["ingest"]
    target = params["data"]["target"]

    dataset = fetch_covtype(
        data_home=config["cache_dir"],
        as_frame=True,
        shuffle=False,
    )

    frame = dataset.data.copy()
    frame[target] = dataset.target.astype("int64")

    zone = frame.loc[frame[config["wilderness_column"]] == 1].copy()

    # Один и тот же порядок позволяет расширять v1, сохраняя все её строки.
    zone = zone.sample(frac=1, random_state=config["seed"])
    n_rows = int(config["n_rows"])

    if not 0 < n_rows <= len(zone):
        raise ValueError(f"n_rows должно быть от 1 до {len(zone)}")

    selected = zone.iloc[:n_rows].reset_index(drop=True)

    if selected.isna().any().any():
        raise ValueError("В данных обнаружены пропуски")

    output = Path(config["raw"])
    output.parent.mkdir(parents=True, exist_ok=True)
    selected.to_csv(output, index=False)

    print(f"Сохранено строк: {len(selected)}")
    print(f"Число признаков: {len(selected.columns) - 1}")
    print(f"Файл: {output}")
    print("Распределение классов:")
    print(selected[target].value_counts().sort_index().to_string())


if __name__ == "__main__":
    main()

Writing src/ingest.py


In [55]:
import subprocess

subprocess.run(["python", "-m", "src.ingest"], check=True)CompletedProcess(args=['python', '-m', 'src.ingest'], returncode=0)$0

CompletedProcess(args=['python', '-m', 'src.ingest'], returncode=0)

In [56]:
from pathlib import Path
import pandas as pd

path = Path("data/raw/covertype.csv")
frame = pd.read_csv(path)

print("Количество строк:", len(frame))
print("Количество признаков:", frame.shape[1] - 1)
print("Размер файла, МБ:", round(path.stat().st_size / 1024**2, 2))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Количество строк: 10000
Количество признаков: 54
Размер файла, МБ: 2.26


In [57]:
import subprocess

subprocess.run(
    ["dvc", "add", "data/raw/covertype.csv"],
    check=True,
)

subprocess.run(["dvc", "push"], check=True)

CompletedProcess(args=['dvc', 'push'], returncode=0)

In [58]:
!dvc status
!git status --short

Data and pipelines are up to date.
A  .dvc/.gitignore
AM .dvc/config
A  .dvcignore
A  .gitignore
A  .pre-commit-config.yaml
M  README.md
A  models/.gitkeep
A  notebooks/.gitkeep
A  pyproject.toml
A  reports/.gitkeep
A  src/__init__.py
A  tests/.gitkeep
A  uv.lock
?? data/
?? params.yaml
?? src/ingest.py


In [59]:
%%writefile params.yaml
ingest:
  cache_dir: data/external
  raw: data/raw/covertype.csv
  wilderness_column: Wilderness_Area_0
  n_rows: 10000
  seed: 42

data:
  target: Cover_Type
  test_size: 0.2
  split_seed: 42

featurize:
  drop:
    - Wilderness_Area_0
    - Wilderness_Area_1
    - Wilderness_Area_2
    - Wilderness_Area_3

Overwriting params.yaml


In [60]:
%%writefile src/utils.py
"""Общие функции чтения параметров и сохранения таблиц."""

from pathlib import Path

import yaml


def load_params():
    return yaml.safe_load(
        Path("params.yaml").read_text(encoding="utf-8")
    )


def save_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)

Writing src/utils.py


In [61]:
%%writefile src/prepare.py
"""Проверка данных и стратифицированное разбиение."""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.utils import load_params, save_csv


def main():
    params = load_params()
    config = params["data"]
    target = config["target"]

    frame = pd.read_csv(params["ingest"]["raw"])

    if frame.empty:
        raise ValueError("Набор данных пуст")

    if target not in frame.columns:
        raise ValueError(f"Отсутствует целевая колонка {target}")

    if not all(pd.api.types.is_numeric_dtype(t) for t in frame.dtypes):
        raise ValueError("Все колонки должны быть числовыми")

    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError("Обнаружены пропуски или бесконечные значения")

    if not frame[target].isin(range(1, 8)).all():
        raise ValueError("Классы Cover_Type должны быть целыми от 1 до 7")

    before = len(frame)
    frame = frame.drop_duplicates().reset_index(drop=True)
    print(f"Удалено полных дубликатов: {before - len(frame)}")

    train, test = train_test_split(
        frame,
        test_size=float(config["test_size"]),
        random_state=int(config["split_seed"]),
        stratify=frame[target],
    )

    save_csv(train, "data/processed/train.csv")
    save_csv(test, "data/processed/test.csv")

    print(f"Обучающая выборка: {len(train)} строк")
    print(f"Тестовая выборка: {len(test)} строк")


if __name__ == "__main__":
    main()

Writing src/prepare.py


In [62]:
%%writefile src/featurize.py
"""Подготовка признаков для обучения и оценки."""

import pandas as pd

from src.utils import load_params, save_csv


def main():
    params = load_params()
    target = params["data"]["target"]
    drop_columns = params["featurize"]["drop"]

    if target in drop_columns:
        raise ValueError("Нельзя удалять целевую переменную")

    for part in ("train", "test"):
        frame = pd.read_csv(f"data/processed/{part}.csv")
        features = frame.drop(columns=drop_columns)

        # Целевая переменная хранится последней колонкой.
        columns = [column for column in features.columns if column != target]
        features = features[columns + [target]]

        save_csv(features, f"data/features/{part}.csv")

        print(
            f"{part}: {len(features)} строк, "
            f"{len(columns)} признаков"
        )


if __name__ == "__main__":
    main()

Writing src/featurize.py


In [63]:
from pathlib import Path

path = Path(".gitignore")
lines = path.read_text(encoding="utf-8").splitlines()

for rule in ("/data/processed/", "/data/features/"):
    if rule not in lines:
        lines.append(rule)

path.write_text("\n".join(lines) + "\n", encoding="utf-8")

131

In [64]:
import subprocess

for module in ("src.prepare", "src.featurize"):
    result = subprocess.run(
        ["python", "-m", module],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Ошибка в модуле {module}")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
Удалено полных дубликатов: 0
Обучающая выборка: 8000 строк
Тестовая выборка: 2000 строк

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
train: 8000 строк, 50 признаков
test: 2000 строк, 50 признаков



In [65]:
import subprocess
import sys

subprocess.run(["uv", "add", "bottleneck>=1.5.0"], check=True)

subprocess.run(
    [
        "uv", "export", "--frozen", "--no-hashes",
        "--group", "dev",
        "-o", "/content/requirements.lock.txt",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", "/content/requirements.lock.txt",
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/requirements.lock.txt'], returncode=0)

In [66]:
from pathlib import Path
import yaml

path = Path("params.yaml")
params = yaml.safe_load(path.read_text(encoding="utf-8"))

params["train"] = {
    "model": "random_forest",
    "n_estimators": 200,
    "max_depth": 20,
    "min_samples_leaf": 1,
    "seed": 42,
    "n_jobs": -1,
}

params["evaluate"] = {
    "min_accuracy": 0.70,
}

path.write_text(
    yaml.safe_dump(params, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("Параметры обучения и оценки добавлены")

Параметры обучения и оценки добавлены


In [67]:
%%writefile src/train.py
"""Обучение классификатора Covertype."""

from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from src.utils import load_params


def main():
    params = load_params()
    config = params["train"]
    target = params["data"]["target"]

    if config["model"] != "random_forest":
        raise ValueError(f"Неизвестная модель: {config['model']}")

    frame = pd.read_csv("data/features/train.csv")
    x_train = frame.drop(columns=[target])
    y_train = frame[target]

    model = RandomForestClassifier(
        n_estimators=int(config["n_estimators"]),
        max_depth=config["max_depth"],
        min_samples_leaf=int(config["min_samples_leaf"]),
        random_state=int(config["seed"]),
        n_jobs=int(config["n_jobs"]),
    )
    model.fit(x_train, y_train)

    output = Path("models/model.joblib")
    output.parent.mkdir(parents=True, exist_ok=True)

    joblib.dump(
        {
            "model": model,
            "features": list(x_train.columns),
        },
        output,
    )

    print(f"Модель обучена на {len(frame)} строках")
    print(f"Сохранена: {output}")


if __name__ == "__main__":
    main()

Writing src/train.py


In [68]:
%%writefile src/evaluate.py
"""Оценка классификатора на тестовой выборке."""

import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

from src.utils import load_params


def main():
    params = load_params()
    target = params["data"]["target"]

    bundle = joblib.load("models/model.joblib")
    frame = pd.read_csv("data/features/test.csv")

    x_test = frame[bundle["features"]]
    y_test = frame[target]
    predictions = bundle["model"].predict(x_test)

    accuracy = float(accuracy_score(y_test, predictions))
    metrics = {
        "accuracy": accuracy,
        "f1_macro": float(
            f1_score(y_test, predictions, average="macro", zero_division=0)
        ),
        "n_test": int(len(frame)),
    }

    output = Path("reports/metrics.json")
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(
        json.dumps(metrics, indent=2) + "\n",
        encoding="utf-8",
    )

    print(json.dumps(metrics, indent=2))

    if accuracy < float(params["evaluate"]["min_accuracy"]):
        raise ValueError("Accuracy ниже заданного порога")


if __name__ == "__main__":
    main()

Writing src/evaluate.py


In [69]:
from pathlib import Path

path = Path(".gitignore")
lines = path.read_text(encoding="utf-8").splitlines()

if "/models/*.joblib" not in lines:
    lines.append("/models/*.joblib")

path.write_text("\n".join(lines) + "\n", encoding="utf-8")

148

In [70]:
import subprocess

for module in ("src.train", "src.evaluate"):
    result = subprocess.run(
        ["python", "-m", module],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(f"Ошибка в модуле {module}")

Модель обучена на 8000 строках
Сохранена: models/model.joblib

{
  "accuracy": 0.852,
  "f1_macro": 0.7304917807163481,
  "n_test": 2000
}



In [71]:
%cd /content/mlops-labs

/content


In [72]:
%%writefile dvc.yaml
stages:
  prepare:
    cmd: python -m src.prepare
    deps:
      - src/__init__.py
      - src/prepare.py
      - src/utils.py
      - data/raw/covertype.csv
    params:
      - ingest.raw
      - data
    outs:
      - data/processed/train.csv
      - data/processed/test.csv

  featurize:
    cmd: python -m src.featurize
    deps:
      - src/__init__.py
      - src/featurize.py
      - src/utils.py
      - data/processed/train.csv
      - data/processed/test.csv
    params:
      - data.target
      - featurize
    outs:
      - data/features/train.csv
      - data/features/test.csv

  train:
    cmd: python -m src.train
    deps:
      - src/__init__.py
      - src/train.py
      - src/utils.py
      - data/features/train.csv
    params:
      - data.target
      - train
    outs:
      - models/model.joblib

  evaluate:
    cmd: python -m src.evaluate
    deps:
      - src/__init__.py
      - src/evaluate.py
      - src/utils.py
      - models/model.joblib
      - data/features/test.csv
    params:
      - data.target
      - evaluate
    metrics:
      - reports/metrics.json:
          cache: false

Writing dvc.yaml


In [73]:
import subprocess

result = subprocess.run(
    ["dvc", "repro"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError("Пайплайн завершился с ошибкой")

'data/raw/covertype.csv.dvc' didn't change, skipping
Running stage 'prepare':
> python -m src.prepare
Удалено полных дубликатов: 0
Обучающая выборка: 8000 строк
Тестовая выборка: 2000 строк
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

Running stage 'featurize':
> python -m src.featurize
train: 8000 строк, 50 признаков
test: 2000 строк, 50 признаков
Updating lock file 'dvc.lock'

Running stage 'train':
> python -m src.train
Модель обучена на 8000 строках
Сохранена: models/model.joblib
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python -m src.evaluate
{
  "accuracy": 0.852,
  "f1_macro": 0.7304917807163481,
  "n_test": 2000
}
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.



In [74]:
import subprocess

result = subprocess.run(
    ["dvc", "repro"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError("Ошибка повторного запуска")

'data/raw/covertype.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'featurize' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.



In [75]:
!dvc dag
!cat reports/metrics.json

+----------------------------+ 
| data/raw/covertype.csv.dvc | 
+----------------------------+ 
               *               
               *               
               *               
          +---------+          
          | prepare |          
          +---------+          
               *               
               *               
               *               
        +-----------+          
        | featurize |          
        +-----------+          
          *         **         
        **            *        
       *               **      
 +-------+               *     
 | train |             **      
 +-------+            *        
          *         **         
           **     **           
             *   *             
         +----------+          
         | evaluate |          
         +----------+          
{
  "accuracy": 0.852,
  "f1_macro": 0.7304917807163481,
  "n_test": 2000
}


In [76]:
import subprocess

subprocess.run(["git", "add", "-A"], check=True)

result = subprocess.run(
    ["pre-commit", "run", "--all-files"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)

check for added large files..............................................Passed
check yaml...............................................................Passed
detect private key.......................................................Passed
fix end of files.........................................................Passed
trim trailing whitespace.................................................Passed
ruff check...............................................................Passed
ruff format..............................................................Failed
- hook id: ruff-format
- files were modified by this hook

2 files reformatted, 6 files left unchanged




In [77]:
!dvc repro

'data/raw/covertype.csv.dvc' didn't change, skipping
Running stage 'prepare':
> python -m src.prepare
Удалено полных дубликатов: 0
Обучающая выборка: 8000 строк
Тестовая выборка: 2000 строк
Updating lock file 'dvc.lock'

Running stage 'featurize':
> python -m src.featurize
train: 8000 строк, 50 признаков
test: 2000 строк, 50 признаков
Updating lock file 'dvc.lock'

Running stage 'train':
> python -m src.train
Модель обучена на 8000 строках
Сохранена: models/model.joblib
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python -m src.evaluate
{
  "accuracy": 0.852,
  "f1_macro": 0.7304917807163481,
  "n_test": 2000
}
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [84]:
import subprocess

subprocess.run(["dvc", "push"], check=True)

CompletedProcess(args=['dvc', 'push'], returncode=0)

коммит первой версии

In [87]:
import os
import subprocess
import tempfile
from pathlib import Path

from google.colab import userdata

env = os.environ.copy()
env["GITHUB_USER"] = userdata.get("GITHUB_USER")
env["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
env["GIT_TERMINAL_PROMPT"] = "0"

with tempfile.TemporaryDirectory() as temp_dir:
    helper = Path(temp_dir) / "git-askpass.sh"
    helper.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    helper.chmod(0o700)
    env["GIT_ASKPASS"] = str(helper)

    commands = [
        ["git", "remote", "set-url", "origin",
         "https://github.com/Anastasia-Marshakova/mlops-labs.git"],
        ["git", "-c", "credential.helper=", "push", "origin", "HEAD"],
        ["git", "-c", "credential.helper=", "push", "origin", "data-v1"],
    ]

    for command in commands:
        result = subprocess.run(
            command,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        output = result.stdout.replace(
            env["GITHUB_TOKEN"], "[TOKEN HIDDEN]"
        )
        print(output)

        if result.returncode != 0:
            raise RuntimeError("Отправка остановлена: смотри сообщение выше")

print("Первая версия отправлена на GitHub")


To https://github.com/Anastasia-Marshakova/mlops-labs.git
   dcf8e57..daf4c9d  HEAD -> main

To https://github.com/Anastasia-Marshakova/mlops-labs.git
 * [new tag]         data-v1 -> data-v1

Первая версия отправлена на GitHub


In [90]:
%cd /content/mlops-labs

from pathlib import Path
import yaml

path = Path("params.yaml")
params = yaml.safe_load(path.read_text(encoding="utf-8"))
params["ingest"]["n_rows"] = 20000

path.write_text(
    yaml.safe_dump(params, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("Размер второй версии: 20000 строк")

/content
Размер второй версии: 20000 строк


In [91]:
import subprocess

commands = [
    ["python", "-m", "src.ingest"],
    ["dvc", "add", "data/raw/covertype.csv"],
    ["dvc", "repro"],
]

for command in commands:
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError("Выполнение остановлено: смотри сообщение выше")

Сохранено строк: 20000
Число признаков: 54
Файл: data/raw/covertype.csv
Распределение классов:
Cover_Type
1     8181
2    11159
5      269
7      391


To track the changes with git, run:

	git add data/raw/covertype.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

'data/raw/covertype.csv.dvc' didn't change, skipping
Running stage 'prepare':
> python -m src.prepare
Удалено полных дубликатов: 0
Обучающая выборка: 16000 строк
Тестовая выборка: 4000 строк
Updating lock file 'dvc.lock'

Running stage 'featurize':
> python -m src.featurize
train: 16000 строк, 50 признаков
test: 4000 строк, 50 признаков
Updating lock file 'dvc.lock'

Running stage 'train':
> python -m src.train
Модель обучена на 16000 строках
Сохранена: models/model.joblib
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python -m src.evaluate
{
  "accuracy": 0.8665,
  "f1_macro": 0.7929977031433043,
  "n_test": 4000
}
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git a

In [92]:
!cat reports/metrics.json

{
  "accuracy": 0.8665,
  "f1_macro": 0.7929977031433043,
  "n_test": 4000
}


In [93]:
import subprocess

subprocess.run(["git", "add", "-A"], check=True)

result = subprocess.run(
    ["pre-commit", "run", "--all-files"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)
print("Код завершения:", result.returncode)

check for added large files..............................................Passed
check yaml...............................................................Passed
detect private key.......................................................Passed
fix end of files.........................................................Passed
trim trailing whitespace.................................................Passed
ruff check...............................................................Passed
ruff format..............................................................Passed

Код завершения: 0


In [94]:
import subprocess

for command in [
    ["dvc", "repro"],
    ["dvc", "push"],
    ["git", "add", "-A"],
    ["git", "commit", "-m", "ЛР1: Covertype v2 — 20000 объектов"],
]:
    subprocess.run(command, check=True)

In [95]:
subprocess.run(
    ["git", "tag", "-a", "data-v2", "-m", "Covertype: 20000 объектов зоны 1"],
    check=True,
)

CompletedProcess(args=['git', 'tag', '-a', 'data-v2', '-m', 'Covertype: 20000 объектов зоны 1'], returncode=0)

In [103]:
import os
import subprocess
import tempfile
from pathlib import Path

from google.colab import userdata

env = os.environ.copy()
env["GITHUB_USER"] = userdata.get("GITHUB_USER")
env["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
env["GIT_TERMINAL_PROMPT"] = "0"

with tempfile.TemporaryDirectory() as temp_dir:
    helper = Path(temp_dir) / "git-askpass.sh"
    helper.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    helper.chmod(0o700)
    env["GIT_ASKPASS"] = str(helper)

    commands = [
        ["git", "remote", "set-url", "origin",
         "https://github.com/Anastasia-Marshakova/mlops-labs.git"],
        ["git", "-c", "credential.helper=", "push", "origin", "HEAD"],
        ["git", "-c", "credential.helper=", "push", "origin", "data-v2"],
    ]

    for command in commands:
        result = subprocess.run(
            command,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        output = result.stdout.replace(
            env["GITHUB_TOKEN"], "[TOKEN HIDDEN]"
        )
        print(output)

        if result.returncode != 0:
            raise RuntimeError("Отправка остановлена: смотри сообщение выше")

print("Вторая версия отправлена на GitHub")


To https://github.com/Anastasia-Marshakova/mlops-labs.git
   ff4889b..4fbc765  HEAD -> main

Everything up-to-date

Вторая версия отправлена на GitHub


In [99]:
import os
from pathlib import Path

os.chdir("/content/mlops-labs")
assert Path(".git").is_dir()
print("Рабочая папка:", Path.cwd())

Рабочая папка: /content/mlops-labs


In [100]:
%%writefile README.md
# Лабораторная работа №1. Воспроизводимый проект: данные и пайплайн

## Автор и вариант

GitHub: Anastasia-Marshakova.
Вариант 10 — Covertype (UCI).

## Цель работы

Создать воспроизводимый ML-проект с зафиксированным окружением,
версионированием данных и пайплайном обработки.

## Данные и задача

Источник: UCI Covertype.
Загрузка выполняется через sklearn.datasets.fetch_covtype.

Задача — многоклассовая классификация типа лесного покрытия.
Целевая переменная — Cover_Type, возможные значения от 1 до 7.

Для обучения выбрана первая лесная зона:
Wilderness_Area_0 = 1 в именах колонок scikit-learn.
Остальные зоны зарезервированы для потока данных в ЛР4.

Версии данных:
- data-v1: 10 000 объектов первой зоны;
- data-v2: 20 000 объектов той же зоны, включая все объекты v1.

Полный загруженный набор находится в data/external/ вне Git и DVC.
Рабочий срез data/raw/covertype.csv хранится через DVC.

## Структура проекта

- src/ — модули обработки, обучения и оценки;
- notebooks/ — рабочий ноутбук;
- tests/ — каталог для тестов;
- data/raw/ — исходный рабочий срез под контролем DVC;
- data/processed/ — обучающая и тестовая выборки;
- data/features/ — подготовленные признаки;
- models/ — обученная модель;
- reports/metrics.json — метрики;
- params.yaml — параметры;
- dvc.yaml и dvc.lock — описание и состояние пайплайна;
- pyproject.toml и uv.lock — окружение.

## Развёртывание в Google Colab

Создать секреты GITHUB_USER, GITHUB_TOKEN, DAGSHUB_USER,
DAGSHUB_TOKEN и разрешить доступ к ним из ноутбука.
Токены не записывать в код и не выводить на экран.

Клонировать публичный репозиторий:

```python
!git clone https://github.com/Anastasia-Marshakova/mlops-labs.git
%cd /content/mlops-labs
```

Установить окружение из сохранённого lock-файла:

```python
!pip install -q uv
!uv export --frozen --no-hashes --group dev -o /content/requirements.lock.txt
!pip install -q -r /content/requirements.lock.txt
```

При воспроизведении uv.lock не пересоздаётся.

## Доступ к хранилищу DVC

Адрес хранилища сохранён в .dvc/config.
В новом клоне необходимо задать локальные ключи:

```python
import subprocess
from google.colab import userdata

token = userdata.get("DAGSHUB_TOKEN")

for key in ("access_key_id", "secret_access_key"):
    result = subprocess.run(
        ["dvc", "remote", "modify", "--local", "origin", key, token],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError("Не удалось настроить локальные ключи DVC")
```

Ключи сохраняются только в .dvc/config.local, исключённом из Git.

## Воспроизведение

```python
!dvc pull
!dvc repro
!cat reports/metrics.json
```

dvc pull загружает данные и результаты этапов.
dvc repro проверяет зависимости и выполняет только необходимые этапы.
Повторный запуск пропускает неизменившиеся этапы.

## Этапы пайплайна

1. prepare — проверка данных, удаление полных дубликатов,
   стратифицированное разбиение 80/20 с фиксированным seed.
2. featurize — удаление четырёх индикаторов лесной зоны:
   внутри выбранной зоны они постоянны.
3. train — обучение RandomForestClassifier.
4. evaluate — расчёт accuracy и F1 macro на тестовой выборке,
   сохранение reports/metrics.json и проверка порога accuracy.

В наборе нет даты наблюдения, поэтому разбиение по времени не используется.
После подготовки остаётся 50 признаков.

## Результаты

| Версия | Всего строк | Размер теста | Accuracy | F1 macro |
|---|---:|---:|---:|---:|
| data-v1 | 10000 | 2000 | 0.8520 | 0.7305 |
| data-v2 | 20000 | 4000 | 0.8665 | 0.7930 |

На второй версии обе метрики выше. Однако тестовые выборки различаются,
поэтому это сравнение не изолирует влияние размера обучающей выборки.

## Переключение версий

Перед переключением рабочее дерево Git должно быть чистым.

```python
!git checkout data-v1
!dvc checkout
!dvc repro
```

Возврат ко второй версии:

```python
!git checkout data-v2
!dvc checkout
!dvc repro
```

Возврат к основной ветке:

```python
!git checkout main
!dvc checkout
```

Проверено: обе версии восстанавливают соответствующее количество строк
и метрики. dvc repro после переключения пропускает все этапы.

Если нужных объектов нет в локальном кеше, выполнить dvc pull.

## Проверки кода

```python
!pre-commit install
!pre-commit run --all-files
```

Настроены Ruff, форматирование, проверка YAML, поиск приватных ключей
и ограничение размера добавляемых файлов 1 МБ.

## Хранение результатов

Git хранит код, параметры, lock-файлы, метафайлы DVC и метрики.
Хранилище DVC содержит рабочие данные, промежуточные таблицы и модель.

Перед отправкой изменений в GitHub выполняется dvc push.

Overwriting README.md


In [101]:
import subprocess

subprocess.run(["git", "add", "README.md"], check=True)

result = subprocess.run(
    ["pre-commit", "run", "--all-files"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)
print("Код завершения:", result.returncode)

check for added large files..............................................Passed
check yaml...............................................................Passed
detect private key.......................................................Passed
fix end of files.........................................................Passed
trim trailing whitespace.................................................Passed
ruff check...............................................................Passed
ruff format..............................................................Passed

Код завершения: 0


In [102]:
subprocess.run(["git", "add", "README.md"], check=True)
subprocess.run(
    ["git", "commit", "-m", "Документация ЛР1: запуск и результаты"],
    check=True,
)

CompletedProcess(args=['git', 'commit', '-m', 'Документация ЛР1: запуск и результаты'], returncode=0)